In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.3 Learning-rate schedules

A fixed learning rate is rarely best. Warm it up (avoid early instability while
the moments are still cold), then cosine-decay it down (anneal into a fine,
settled minimum).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pocketlm import cosine_lr

total = 200
lrs = [cosine_lr(s, warmup=20, total=total, base_lr=3e-4) for s in range(total)]
plt.plot(lrs)
plt.xlabel("step")
plt.ylabel("learning rate")
plt.title("warmup + cosine decay")
plt.show()
print("start:", round(lrs[0], 7), " peak:", round(lrs[19], 7),
      " end:", round(lrs[-1], 8))

The shape is a ramp then a gentle slope to near zero. This single schedule covers
the vast majority of from-scratch training runs.

In [ ]:
assert lrs[0] < lrs[19]      # warming up
assert lrs[-1] < lrs[19]     # decaying